In [16]:
import os
sim_type, snapnum='DM', 237
# Load halo density profiles
halos_dir = f'../result/DMhalo_density_profiles_phys/MTNG/{sim_type}-Arepo/MTNG-L500-4320-A/snap_{snapnum}/final_densities/'
halos_list = os.listdir(halos_dir)
# sort the list
halos_list = sorted(halos_list)
print(halos_list)
# Get the bin starts and ends
bin_starts = [float(halo.split('-')[1])/10 for halo in halos_list]
bin_ends = [float(halo.split('-')[2].split('.')[0])/10 for halo in halos_list]
print(bin_starts)
print(bin_ends)  

['bin-30-35.npy', 'bin-35-40.npy', 'bin-40-45.npy', 'bin-45-50.npy']
[3.0, 3.5, 4.0, 4.5]
[3.5, 4.0, 4.5, 5.0]


In [17]:
import illustris_python as il

sim = f'{sim_type}-Arepo/MTNG-L500-4320-A'
basePath = f'/virgotng/mpa/MTNG/{sim}/output/'
snap_all_mass = il.groupcat.loadHalos(basePath, snapnum, fields='Group_M_Mean200')
print(snap_all_mass.shape)

(79058617,)


In [18]:
# Load FPG mass
import numpy as np
# This is physical mass!
FPGrMass = np.load(f'../result/DMhalo_mass_table_new/{sim_type}-Arepo/MTNG-L500-4320-A/snap_{snapnum}_FPGrMass.npy')
print(FPGrMass.shape)

(75324756,)


In [19]:
# Load FPG idx
FPGr = np.load(f'../result/DMhalo_mass_table_new/{sim_type}-Arepo/MTNG-L500-4320-A/snap_{snapnum}_FPGr.npy')
print(FPGr.shape)

(75324756,)


In [26]:
# Load the accretion rate
accretion_rates = np.load(f'../result/DMhalo_mass_table_new/{sim_type}-Arepo/MTNG-L500-4320-A/accretion_rates.npy')

In [27]:
print(accretion_rates.shape)

(75324756, 5)


In [ ]:
# Load the data
halo_M, halo_R, bins, densities, accretions = [], [], [], [], []
for fname, start, end in zip(halos_list, bin_starts, bin_ends):
    print(fname)
    
    data = np.load(os.path.join(halos_dir, fname), allow_pickle=True).item()
    halo_M.append(data['halo_M_Mean200']) # This is physical mass!
    halo_R.append(data['halo_R_Mean200'])
    bins.append(data['radial_bins'])
    densities.append(data['densities'])
    print(data['halo_M_Mean200'].shape)
    
    # Get the halo snap indices
    subset_idx = np.where((snap_all_mass >= 10**start) & (snap_all_mass < 10**end))[0]
    print(subset_idx.shape)
    
   
    accret = []
    for idx in subset_idx[:4]:
        # Check the masses
        idx_in_FPGr = np.where(FPGr == idx)[0]
        print(FPGrMass[idx_in_FPGr].shape, snap_all_mass[idx])
        print(np.all(FPGrMass[idx_in_FPGr] == FPGrMass[idx_in_FPGr][0]), FPGrMass[idx_in_FPGr][0])
        
        # Extract the corresponding accretion rates and get the mean
        print(accretion_rates[idx_in_FPGr].shape)
        mean_accret = np.mean(accretion_rates[idx_in_FPGr])
        print(mean_accret)
        accret.append(mean_accret)
    print('')
    
    # Check the added accertion rates shape
    accretions.append(accret)
accretions = np.concatenate(accretions)
print(accretions.shape)

bin-30-35.npy
(38320,)
(38320,)

bin-35-40.npy
(11091,)
(11091,)

bin-40-45.npy
(2319,)
(2319,)

bin-45-50.npy
(266,)
(266,)

(16,)
